# 하이브리드 검색과 리랭킹



## 하이브리드 검색
키워드 검색(Keyword Search)과 의미 기반 검색(Semantic Search)의 장점을 결합하여 검색 정확도를 높이는 기법

RAG(Retrieval-Augmented Generation) 시스템 구축 시, 단순히 벡터 검색만 사용할 경우 고유명사나 정확한 용어 매칭이 어려울 수 있는데, 하이브리드 서치는 이를 보완(BM25 → 키워드 기반 정확한 검색, Vector Search → 의미 기반 검색)

### 키워드 검색(Sparse Retriever):

- BM25 알고리즘이 대표적: 단어의 빈도와 역문서 빈도를 기반으로 작동하며, 정확한 단어 매칭에 강합니다.

### 의미 검색(Dense Retriever):

- 임베딩(Embedding) 벡터 유사도(Cosine Similarity 등)를 기반: 단어가 달라도 문맥적 의미가 유사한 문서를 찾는 데 강합니다.

In [ ]:
# 하이브리드 검색과 리랭킹을 구현하기 위해 필요한 라이브러리들을 설치합니다
# - langchain: LLM 애플리케이션 구축을 위한 핵심 프레임워크
# - langchain-community: 커뮤니티에서 제공하는 다양한 통합 모듈 (BM25Retriever 등)
# - rank_bm25: BM25 알고리즘 구현 라이브러리 (키워드 기반 검색)
# - langchain_openai: OpenAI 모델과의 통합 모듈
# - faiss-cpu: 벡터 검색 라이브러리 (의미 기반 검색)
!pip install langchain langchain-community rank_bm25 langchain_openai faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


> LangChain에서는 EnsembleRetriever 클래스를 사용하여 매우 쉽게 하이브리드 서치를 구현할 수 있습니다. 가장 일반적인 조합은 **BM25(키워드)**와 **VectorStore(의미)**의 결합입니다.

In [ ]:
# 시스템과 상호작용하기 위한 모듈 임포트
import getpass  # 비밀번호 입력처럼 안전하게 사용자 입력을 받기 위한 모듈
import os       # 운영체제와 상호작용, 환경 변수 설정을 위한 모듈

# 사용자로부터 OpenAI API 키를 안전하게 입력받아 환경 변수에 설정합니다
# getpass.getpass() 함수는 입력 내용이 화면에 표시되지 않도록 합니다 (보안 강화)
# 환경 변수에 설정된 API 키는 이후 LangChain의 OpenAI 관련 모듈에서 자동으로 사용됩니다
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# 코드 실행 흐름:
# 1. 필요한 라이브러리 설치 (인터넷 연결 필요)
# 2. OpenAI API 키 입력 요청 및 환경 변수 설정
# 3. 이후 코드에서 이 환경 변수를 통해 OpenAI 서비스에 접근

# 설치된 라이브러리들의 역할:
# - rank_bm25: 전통적인 키워드 기반 검색 알고리즘인 BM25를 제공합니다
# - faiss-cpu: Facebook AI Research에서 개발한 고속 벡터 검색 라이브러리로,
#   임베딩된 문서들의 유사도 검색에 사용됩니다
# - langchain-community: BM25Retriever와 같은 커뮤니티 제공 검색기를 포함합니다

# 하이브리드 검색의 개념:
# - 키워드 검색(BM25): 정확한 단어 매칭에 강점, 고유명사나 특정 용어 검색에 적합
# - 의미 검색(벡터 검색): 문맥적 의미 유사도에 기반, 동의어나 관련 개념 검색에 적합
# - 두 방법을 결합하여 더 정확하고 강력한 검색 시스템 구축

# 리랭킹의 개념:
# - 1차 검색으로 후보 문서들을 추출한 후, 더 정교한 모델로 재정렬하는 과정
# - 속도가 빠른 검색기로 넓은 후보군을 선별 → 정확도 높은 리랭커로 최종 순위 결정
# - RAG 시스템의 검색 정확도를 크게 향상시킬 수 있는 기술

# 이 설정이 완료되면 다음 단계에서:
# 1. BM25Retriever와 FAISS 벡터 저장소를 초기화하고
# 2. 두 검색 결과를 결합하는 하이브리드 검색기를 구현하게 됩니다

Enter your OpenAI API key: ··········


In [ ]:
# LangChain에서 제공하는 다양한 검색기(Retriever)와 도구들을 임포트합니다
from langchain_community.retrievers import BM25Retriever  # BM25 알고리즘 기반 키워드 검색기
from langchain_community.vectorstores import FAISS  # FAISS 벡터 데이터베이스 (의미 기반 검색)
from langchain_openai import OpenAIEmbeddings  # OpenAI의 임베딩 모델 (텍스트를 벡터로 변환)
from langchain_core.documents import Document  # 문서를 표현하는 표준 클래스
from langchain_core.runnables import RunnableLambda  # 함수를 Runnable 객체로 변환하는 래퍼

# 1. 샘플 데이터 준비
# 테스트를 위한 간단한 문서 목록을 생성합니다
# 각 문서는 문자열로 표현되며, 실제 애플리케이션에서는 파일, 데이터베이스, 웹 페이지 등에서 로드할 수 있습니다
docs_list = [
    "아이폰 15의 배터리 수명은 20시간입니다.",  # 문서 1: 아이폰 배터리 정보
    "갤럭시 S24는 AI 기능이 탑재되어 있습니다.",  # 문서 2: 갤럭시 AI 기능 정보
    "사과는 맛있는 과일입니다.",  # 문서 3: 사과 과일 정보 (모호성 테스트용)
    "배터리 절약 모드를 켜면 사용 시간이 늘어납니다."  # 문서 4: 배터리 절약 팁
]

# 2. BM25 검색기 생성 및 설정
# BM25Retriever.from_texts() 메서드로 문서 목록에서 BM25 검색기 인스턴스를 생성합니다
# BM25는 전통적인 키워드 기반 검색 알고리즘으로, TF-IDF의 발전된 형태입니다
# 특징: 정확한 단어 매칭에 강하며, 고유명사나 특정 용어 검색에 적합합니다
bm25 = BM25Retriever.from_texts(docs_list)
# k=2로 설정하여 BM25 검색에서 상위 2개의 결과만 반환하도록 합니다
# 이는 검색 속도와 결과 품질 사이의 균형을 위한 설정입니다
bm25.k = 2

# 3. 벡터 검색기 생성 및 설정
# OpenAIEmbeddings()를 사용하여 텍스트 임베딩 모델을 초기화합니다
# 이 모델은 텍스트를 고차원 벡터로 변환합니다 (예: 1536차원)
embedding = OpenAIEmbeddings()
# FAISS.from_texts() 메서드로 문서 목록과 임베딩 모델을 사용하여 벡터 저장소를 생성합니다
# FAISS는 Facebook AI Research에서 개발한 고속 벡터 검색 라이브러리입니다
# 내부적으로 모든 문서를 임베딩하여 벡터 인덱스를 구축합니다
vectorstore = FAISS.from_texts(docs_list, embedding)
# as_retriever() 메서드로 벡터 저장소를 검색기 인터페이스로 변환합니다
# search_kwargs={"k": 2}로 설정하여 벡터 검색에서도 상위 2개의 결과만 반환합니다
vec = vectorstore.as_retriever(search_kwargs={"k": 2})

# 4. 하이브리드 검색 함수 정의
# hybrid_search 함수는 쿼리를 입력받아 BM25와 벡터 검색의 결과를 결합합니다
def hybrid_search(query):
    # BM25 검색 실행: invoke() 메서드로 쿼리를 전달하여 BM25 알고리즘으로 문서 검색
    # 반환값은 Document 객체의 리스트입니다 (상위 k개)
    bm25_docs = bm25.invoke(query)

    # 벡터 검색 실행: invoke() 메서드로 쿼리를 전달하여 임베딩 유사도 기반 문서 검색
    # 반환값은 Document 객체의 리스트입니다 (상위 k개)
    vec_docs = vec.invoke(query)

    # 중복 제거 + 점수 재정렬을 위한 딕셔너리
    # 키: 문서 내용, 값: 누적 점수 (여러 검색기에서의 점수 합산)
    doc_dict = {}

    # BM25 결과 처리: 각 문서에 가중치 0.5 부여
    # get() 메서드는 키가 존재하면 해당 값을, 없으면 기본값 0을 반환합니다
    for doc in bm25_docs:
        doc_dict[doc.page_content] = doc_dict.get(doc.page_content, 0) + 0.5

    # 벡터 결과 처리: 각 문서에 가중치 0.5 부여
    # BM25와 벡터 검색 모두에서 나온 문서는 점수가 1.0이 됩니다 (0.5 + 0.5)
    for doc in vec_docs:
        doc_dict[doc.page_content] = doc_dict.get(doc.page_content, 0) + 0.5

    # 점수 높은 순으로 정렬
    # items()로 (문서내용, 점수) 튜플 리스트를 얻고, 점수(x[1])를 기준으로 내림차순 정렬
    ranked_docs = sorted(doc_dict.items(), key=lambda x: x[1], reverse=True)

    # Document 형태로 변환하여 반환
    # 정렬된 결과에서 문서 내용만 추출하여 Document 객체로 재생성합니다
    return [Document(page_content=content) for content, _ in ranked_docs]

# 5. LCEL Runnable로 감싸기
# RunnableLambda는 일반 함수를 LangChain의 Runnable 인터페이스로 변환합니다
# 이를 통해 hybrid_search 함수를 다른 LangChain 컴포넌트와 연결하거나 체인에 포함시킬 수 있습니다
hybrid_retriever = RunnableLambda(hybrid_search)

# 하이브리드 검색의 작동 원리:
# 1. 동일한 쿼리에 대해 BM25(키워드)와 벡터(의미) 검색을 병렬로 실행합니다
# 2. 각 검색 방법에서 얻은 결과에 가중치를 부여하여 점수를 합산합니다 (여기서는 균등 가중치 0.5씩)
# 3. 중복 문서를 제거하고 통합 점수로 재정렬합니다
# 4. 최종적으로 점수가 높은 순서로 문서를 반환합니다

# 가중치 조정의 의미:
# - BM25 가중치 증가 → 키워드 정확도 중시 (고유명사, 정확한 용어 검색 강화)
# - 벡터 가중치 증가 → 의미 유사도 중시 (동의어, 관련 개념 검색 강화)
# - 현재는 0.5:0.5로 균형 잡힌 설정

# 이 하이브리드 검색기의 장점:
# 1. 키워드 검색의 정확성과 의미 검색의 맥락 이해력을 결합
# 2. "배터리" 쿼리 시: BM25는 "배터리"가 포함된 문서, 벡터는 "수명", "사용 시간" 등 관련 문서 모두 검색
# 3. "사과" 쿼리 시: BM25는 "사과는 맛있는 과일" 문서, 벡터는 "아이폰" 문서도 검색 가능 (문맥에 따라)

# 다음 단계에서는 이 hybrid_retriever를 사용하여 실제 검색을 실행하게 됩니다

In [ ]:
# 6. 검색 실행 및 결과 출력
# 검색 쿼리를 정의합니다: "스마트폰 배터리 성능"
# 이 쿼리는 스마트폰의 배터리 성능에 대한 정보를 찾고자 하는 사용자의 질의입니다
query = "스마트폰 배터리 성능"

# 하이브리드 검색기 실행
# hybrid_retriever.invoke(query)는 RunnableLambda로 감싸진 hybrid_search 함수를 호출합니다
# 내부적으로 다음 과정이 실행됩니다:
# 1. BM25 검색기 실행: "스마트폰 배터리 성능" 쿼리로 키워드 기반 검색
# 2. 벡터 검색기 실행: 같은 쿼리로 의미 기반 검색
# 3. 두 결과를 가중치 합산 및 정렬
# 4. 정렬된 Document 객체 리스트 반환
docs = hybrid_retriever.invoke(query)

# 검색 결과 출력
# for 루프를 사용하여 검색된 각 문서를 순회하며 내용을 출력합니다
# enumerate를 사용하지 않고 간단하게 하이픈(-)으로 목록을 표시합니다
for d in docs:
    # 각 문서의 page_content(문서 내용)을 출력합니다
    print("-", d.page_content)

# 검색 과정 상세 설명:
# 1. BM25 검색기 동작:
#    - "스마트폰 배터리 성능" 쿼리를 단어로 분리: ["스마트폰", "배터리", "성능"]
#    - 각 문서에서 이 단어들의 빈도(TF)와 문서 집합에서의 역빈도(IDF)를 계산
#    - 점수 계산 후 상위 2개(k=2) 문서 선택
#    - 예상 결과:
#        a. "아이폰 15의 배터리 수명은 20시간입니다." (배터리 단어 일치)
#        b. "배터리 절약 모드를 켜면 사용 시간이 늘어납니다." (배터리 단어 일치)
#    - "갤럭시 S24" 문서는 AI 관련이므로 점수 낮음
#    - "사과는 맛있는 과일입니다." 문서는 관련 없음

# 2. 벡터 검색기 동작:
#    - OpenAI 임베딩 모델이 "스마트폰 배터리 성능" 쿼리를 벡터로 변환
#    - FAISS가 미리 구축한 문서 벡터 인덱스에서 코사인 유사도 계산
#    - 유사도가 높은 상위 2개 문서 선택
#    - 예상 결과:
#        a. "아이폰 15의 배터리 수명은 20시간입니다." (의미상 가장 가까움)
#        b. "배터리 절약 모드를 켜면 사용 시간이 늘어납니다." (배터리 관련)
#    - "갤럭시 S24" 문서도 스마트폰 관련이지만 AI 기능 강조로 배터리와 직접적 관련 낮음
#    - "사과는 맛있는 과일입니다." 문서는 의미상 거리가 멈

# 3. 하이브리드 통합 과정:
#    - BM25 결과 2개 + 벡터 결과 2개 = 총 4개 문서 후보
#    - 중복 문서 제거: 두 방법 모두에서 "아이폰 15..."와 "배터리 절약 모드..." 문서가 선택됨
#    - 점수 계산:
#        * "아이폰 15의 배터리 수명은 20시간입니다.": BM25(0.5) + 벡터(0.5) = 1.0
#        * "배터리 절약 모드를 켜면 사용 시간이 늘어납니다.": BM25(0.5) + 벡터(0.5) = 1.0
#        * 만약 다른 문서가 하나의 방법에서만 선택되었다면 점수 0.5
#    - 점수 기준 내림차순 정렬: 두 문서 모두 1.0점으로 동일 점수
#    - 동점일 경우 정렬 안정성에 따라 원래 순서 유지 또는 무작위 정렬될 수 있음

# 4. 최종 출력 예상:
#    - "아이폰 15의 배터리 수명은 20시간입니다."
#    - "배터리 절약 모드를 켜면 사용 시간이 늘어납니다."
#    (순서는 구현에 따라 다를 수 있음)

# 하이브리드 검색의 효과:
# 1. 키워드 일치 강화: "배터리"라는 정확한 단어가 있는 문서 확보
# 2. 의미 확장: "스마트폰"과 "아이폰"의 의미적 연결을 통해 관련 문서 검색
# 3. 정확도 향상: 단일 검색 방법보다 더 포괄적이고 정확한 결과 제공

# 실제 애플리케이션에서의 활용:
# - RAG 시스템에서 더 정확한 문서 검색을 통해 LLM의 응답 품질 향상
# - 검색 엔진에서 검색 결과의 다양성과 정확성 균형
# - 고객 지원 시스템에서 관련 문서 더 잘 찾기

# 주의사항:
# - 샘플 데이터가 적어 모든 검색 시나리오를 테스트하기 어려움
# - 실제 시스템에서는 수천, 수만 개의 문서를 대상으로 검색
# - 가중치(0.5, 0.5)는 도메인과 데이터 특성에 따라 실험적으로 조정 필요
# - k값(검색 결과 수)도 성능과 정확도 균형 고려하여 설정

# 다음으로는 이 하이브리드 검색 결과를 LLM에 전달하여 응답을 생성하는
# 완전한 RAG 시스템으로 확장할 수 있습니다:
# 1. 검색된 문서들을 LLM에 컨텍스트로 제공
# 2. LLM이 문서 정보를 바탕으로 정확한 응답 생성
# 3. 하이브리드 검색으로 인해 더 관련성 높은 문서 제공 → 더 정확한 응답

- 배터리 절약 모드를 켜면 사용 시간이 늘어납니다.
- 사과는 맛있는 과일입니다.
- 아이폰 15의 배터리 수명은 20시간입니다.


> BM25 비중을 줄이고 싶으면 수치 ↓
Vector 비중을 높이고 싶으면 수치 ↑

예:

더 의미 검색 중심 → BM25: 0.2, Vector: 0.8

더 키워드 중심 → BM25: 0.6, Vector: 0.4

## 리랭킹(Re-ranking): "속도 vs 정확도"의 타협점

- Retrieval 단계 (Bi-Encoder):

    - 우리가 지금까지 배운 Vector Search나 BM25입니다.
    - 특징: 미리 계산된 벡터를 비교하므로 속도가 엄청 빠릅니다. 하지만 질문과 문서의 디테일한 관계를 완벽히 파악하지는 못합니다. (대략적인 유사도)
    - 비유: 서류 전형 (수천 명의 지원자 중 스펙이 비슷한 50명을 빠르게 추려냄)

### Re-ranking 단계 (Cross-Encoder):

Retrieval이 뽑아온 상위 문서(예: 50개)와 질문을 **하나의 쌍(Pair)**으로 묶어서 AI 모델에 직접 넣습니다.
- 특징: 두 문장의 관계를 깊이 있게 분석하므로 정확도가 매우 높습니다. 하지만 연산량이 많아 속도가 느립니다. (그래서 전체 문서가 아니라 상위 50개만 검사합니다.)
- 비유: 심층 면접 (서류 통과자 50명을 한 명씩 자세히 인터뷰하여 최종 3명을 선발)


- 요약
    - Vector DB로 빠르고 넓게 후보를 찾고(Candidate Generation)
    - Cross-Encoder로 정밀하게 순위를 다시 매깁니다(Re-ranking).

In [ ]:
# 리랭킹(Re-ranking)을 구현하기 위해 필요한 라이브러리들을 설치합니다
# - langchain-huggingface: HuggingFace 모델들을 LangChain에서 사용하기 위한 통합 모듈
# - sentence-transformers: 문장 임베딩 및 문장 쌍 분류 작업을 위한 라이브러리
# - rank_bm25: BM25 알고리즘 구현 (키워드 기반 검색)
!pip install langchain-huggingface sentence-transformers rank_bm25

In [ ]:
# 시스템과 상호작용하기 위한 모듈 임포트
import getpass  # 비밀번호 입력처럼 안전하게 사용자 입력을 받기 위한 모듈
import os       # 운영체제와 상호작용, 환경 변수 설정을 위한 모듈

# 사용자로부터 HuggingFace 토큰을 안전하게 입력받아 환경 변수에 설정합니다
# HuggingFace 토큰은 HuggingFace 모델 허브에서 gated model(접근 제한 모델)을 다운로드하거나
# Inference API를 사용할 때 필요할 수 있습니다
# getpass.getpass() 함수는 입력 내용이 화면에 표시되지 않도록 합니다
os.environ["HF_TOKEN"] = getpass.getpass("HuggingFace Token: ")

# 코드 실행 흐름:
# 1. 필요한 라이브러리 설치 (인터넷 연결 필요)
# 2. HuggingFace 토큰 입력 요청 및 환경 변수 설정
# 3. 이후 코드에서 이 환경 변수를 통해 HuggingFace 모델에 접근

# 설치된 라이브러리들의 역할:
# - langchain-huggingface: HuggingFace의 다양한 모델(임베딩, 크로스 인코더 등)을
#   LangChain 컴포넌트로 사용할 수 있게 합니다
# - sentence-transformers: 문장 임베딩 모델과 크로스 인코더 모델을 제공합니다
# - rank_bm25: 이전과 동일하게 BM25 알고리즘을 제공합니다

# 리랭킹(Re-ranking)의 개념적 이해:
# 1단계 - 후보 생성(Candidate Generation):
#   - 벡터 검색(FAISS)이나 키워드 검색(BM25)으로 빠르게 수백~수천 개의 문서 중
#     상위 N개(예: 50개)의 후보 문서를 추출합니다
#   - 속도는 빠르지만 정밀도가 상대적으로 낮습니다 (대략적인 유사도만 측정)
#   - Bi-Encoder 방식: 질문과 문서를 각각 독립적으로 임베딩하여 유사도 계산

# 2단계 - 리랭킹(Re-ranking):
#   - 1단계에서 추출한 상위 N개의 후보 문서를 대상으로 더 정밀한 재정렬을 수행합니다
#   - Cross-Encoder 방식을 사용: 질문과 문서를 하나의 입력으로 묶어서
#     직접 관계 점수를 계산 (예: [CLS] 질문 [SEP] 문서 [SEP])
#   - 속도는 느리지만(연산량 많음) 정밀도가 매우 높습니다
#   - 질문-문서 쌍의 의미적 관계를 깊이 있게 분석할 수 있습니다

# 비유적 설명:
# - 1단계(후보 생성): 서류 전형 - 수천 명의 지원자 중 스펙이 비슷한 50명을 빠르게 추려냄
# - 2단계(리랭킹): 심층 면접 - 서류 통과자 50명을 한 명씩 자세히 인터뷰하여 최종 3명 선발

# 리랭킹의 장점:
# - 정확도 향상: 단순 유사도 계산보다 질문과 문서의 관계를 더 잘 이해
# - 모호성 해소: "사과"라는 단어가 과일인지 기업(Apple)인지 문맥에 맞게 판단
# - 관련성 평가: 질문에 정말로 답변할 수 있는 문서인지 더 정확히 평가

# 리랭킹의 단점:
# - 속도 저하: 각 질문-문서 쌍을 모델에 별도로 전달해야 하므로 연산량 많음
# - 비용 증가: 더 큰 모델을 사용하므로 컴퓨팅 자원 더 필요

# 일반적인 리랭킹 워크플로우:
# 1. 쿼리(질문) 입력
# 2. 벡터 검색이나 BM25로 상위 50개 문서 후보 추출
# 3. 각 후보 문서에 대해 Cross-Encoder로 질문-문서 쌍 점수 계산
# 4. Cross-Encoder 점수로 문서 재정렬
# 5. 상위 3-5개 문서만 최종적으로 RAG의 컨텍스트로 사용

# 이 설정이 완료되면 다음 단계에서:
# 1. BM25와 벡터 검색을 결합한 하이브리드 검색 구현
# 2. Cross-Encoder 모델을 사용한 리랭킹 구현
# 3. 하이브리드 검색 + 리랭킹의 최종 검색 시스템 구축

HuggingFace Token: ··········


In [ ]:
# 리랭킹 시스템 구축을 위해 필요한 모듈들을 임포트합니다
from langchain_community.retrievers import BM25Retriever  # BM25 키워드 검색기
from langchain_community.vectorstores import FAISS  # FAISS 벡터 데이터베이스
from langchain_huggingface import HuggingFaceEmbeddings  # HuggingFace 임베딩 모델
from langchain_community.cross_encoders import HuggingFaceCrossEncoder  # 크로스 인코더 리랭커
from langchain_core.documents import Document  # 표준 문서 클래스
from langchain_core.runnables import RunnableLambda  # 함수를 Runnable로 변환

# ------------------------------
# 1. 문서 준비
# ------------------------------
# 테스트를 위한 레스토랑 추천 문서들을 준비합니다
# 각 문서는 특정 지역의 맛집 정보를 담고 있으며, 지역과 음식 종류가 다양하게 포함되어 있습니다
# 목표: "서울 강남역 파스타 맛집 추천" 쿼리에 가장 적합한 문서 찾기
texts = [
    "강남역에서 가까운 스테이크 맛집 추천 리스트입니다. 분위기 좋은 고기 레스토랑이 많습니다.",  # 문서1: 강남역 스테이크
    "서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다. 크림, 토마토 파스타가 유명합니다.",   # 🎯 정답 (가장 적합)
    "강남역 맛집 전체 가이드입니다. 파스타부터 고기, 한식까지 다양한 식당을 포함합니다.",  # 문서3: 종합 가이드
    "서울 홍대 파스타 맛집 추천입니다. 강남역과는 거리가 꽤 있습니다.",  # 문서4: 홍대 파스타 (지역 다름)
    "부산 남천동 파스타 맛집 리스트입니다. 부산 지역 이탈리안 맛집 모음입니다.",  # 문서5: 부산 파스타 (지역 다름)
    "서울 강남역 카페 추천 리스트입니다. 디저트와 커피로 유명한 곳이 많습니다."  # 문서6: 강남역 카페 (음식 종류 다름)
]

# ------------------------------
# 2. BM25 Retriever (키워드 검색기)
# ------------------------------
# BM25Retriever를 문서 목록으로 초기화합니다
# k=3으로 설정하여 상위 3개의 결과만 반환하도록 합니다
# BM25는 키워드 매칭에 강점이 있어 "강남역", "파스타" 같은 정확한 단어를 찾는데 효과적입니다
bm25 = BM25Retriever.from_texts(texts)
bm25.k = 3

# ------------------------------
# 3. Vector Retriever (의미 검색기)
# ------------------------------
# 한국어 특화 임베딩 모델 'jhgan/ko-sroberta-multitask'를 사용합니다
# 이 모델은 한국어 텍스트의 의미를 잘 포착하도록 사전 학습되었습니다
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
# FAISS 벡터 저장소를 문서들과 임베딩 모델로 생성합니다
vectorstore = FAISS.from_texts(texts, embeddings)
# 벡터 저장소를 검색기로 변환하고 k=3으로 설정합니다
vec = vectorstore.as_retriever(search_kwargs={"k": 3})

# ------------------------------
# 4. 하이브리드 검색 (BM25 + 벡터)
# ------------------------------
# 하이브리드 검색 함수 정의: 키워드 검색과 의미 검색의 결과를 통합합니다
def hybrid_search(query: str):
    # BM25 검색 실행 - 키워드 기반 검색
    bm25_docs = bm25.invoke(query)
    # 벡터 검색 실행 - 의미 기반 검색
    vec_docs = vec.invoke(query)

    # 문서 점수를 저장할 딕셔너리 (문서 내용: 점수)
    scores = {}

    # BM25 결과 처리: 각 문서에 0.5점 부여
    # get() 메서드는 문서가 이미 있으면 현재 점수를, 없으면 0을 반환합니다
    for d in bm25_docs:
        scores[d.page_content] = scores.get(d.page_content, 0) + 0.5

    # Vector 결과 처리: 각 문서에 0.5점 추가 부여
    # 두 검색 방법 모두에서 나타난 문서는 1.0점(0.5+0.5)을 받게 됩니다
    for d in vec_docs:
        scores[d.page_content] = scores.get(d.page_content, 0) + 0.5

    # 점수를 기준으로 내림차순 정렬 (높은 점수 순)
    # items()로 (문서내용, 점수) 튜플 리스트 생성 → 점수(x[1]) 기준 정렬
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # 정렬된 문서 내용을 Document 객체로 변환하여 반환
    return [Document(page_content=c) for c, _ in ranked]

# 하이브리드 검색 함수를 RunnableLambda로 래핑하여 LangChain 체인에서 사용 가능하게 합니다
hybrid_retriever = RunnableLambda(hybrid_search)

# 하이브리드 검색의 기대 효과:
# 1. BM25의 장점: "강남역", "파스타" 같은 정확한 키워드 매칭
# 2. 벡터 검색의 장점: "이탈리안 레스토랑", "맛집" 같은 의미적 유사도
# 3. 통합 점수: 두 방법의 결과를 조합하여 더 강력한 검색 결과 제공

# 예상 검색 동작 (쿼리: "서울 강남역 파스타 맛집 추천"):
# - BM25는 "강남역", "파스타" 키워드가 있는 문서들을 우선 선택
# - 벡터 검색은 "서울", "맛집", "추천"의 의미적 유사도를 고려
# - 두 방법 모두 문서2(정답)를 높은 순위로 선택할 가능성 높음
# - 하이브리드 점수: 문서2는 1.0점(두 방법 모두에서 선택됨), 다른 문서는 0.5점(한 방법에서만 선택됨)

# 다음 단계에서는 이 하이브리드 검색 결과를 Cross-Encoder로 리랭킹할 것입니다:
# 1. 하이브리드 검색으로 후보 문서 선정 (예: 상위 3-5개)
# 2. Cross-Encoder로 질문-문서 쌍의 관련성 점수 계산
# 3. 최종적으로 가장 관련성 높은 문서 선택

In [ ]:
# ------------------------------
# 5. Cross-Encoder를 사용한 직접 Re-ranking 구현
# ------------------------------
# HuggingFaceCrossEncoder를 초기화하여 리랭킹 모델을 생성합니다
# model_name="BAAI/bge-reranker-base": BAAI(Big Science AI)에서 개발한 bge-reranker-base 모델 사용
# 이 모델은 문장 쌍의 관련성을 평가하는 데 특화된 크로스 인코더 모델입니다
# 특징: 질문과 문서를 하나의 입력으로 받아 직접 관련성 점수를 계산
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")  # 문장 비교 특화 모델

# 리랭킹 함수 정의: Cross-Encoder를 사용하여 문서들을 재정렬합니다
def rerank_with_cross_encoder(query: str, docs, top_n: int = 1):
    # (query, 문서) 쌍 만들기
    # 각 문서와 질문을 쌍(pair)으로 묶어 리스트 생성
    # 예: [(query, 문서1내용), (query, 문서2내용), ...]
    # Cross-Encoder는 이 쌍들을 한 번에 처리하여 각 쌍의 관련성 점수를 계산합니다
    pairs = [(query, d.page_content) for d in docs]

    # Cross-Encoder로 점수 계산
    # cross_encoder.score(pairs)는 각 (질문, 문서) 쌍에 대한 관련성 점수를 리스트로 반환합니다
    # 점수는 일반적으로 0-1 사이의 값이나 모델에 따라 다른 범위일 수 있으며, 높을수록 관련성이 높음
    scores = list(cross_encoder.score(pairs))   # List[float] 형식의 점수 리스트

    # 점수 기준 내림차순 정렬
    # zip(docs, scores)로 문서 객체와 점수를 짝지어 튜플 리스트 생성
    # key=lambda x: x[1]로 점수(두 번째 요소)를 기준으로 정렬, reverse=True로 내림차순
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    # 상위 top_n개의 문서만 반환 (점수가 높은 순서로)
    return [doc for doc, _ in ranked[:top_n]]

# Cross-Encoder의 작동 원리:
# 1. Bi-Encoder vs Cross-Encoder:
#    - Bi-Encoder(이전의 임베딩 모델): 질문과 문서를 각각 독립적으로 임베딩 → 코사인 유사도 계산
#    - Cross-Encoder(리랭킹 모델): 질문과 문서를 하나의 입력으로 결합 → 직접 관련성 점수 계산
# 2. 입력 형식: [CLS] 질문 [SEP] 문서 [SEP] (트랜스포머 모델의 표준 형식)
# 3. 출력: 관련성 점수 (예: 0.95 = 매우 관련 있음, 0.05 = 관련 없음)

# 리랭킹 프로세스의 단계:
# 1. 하이브리드 검색 등으로 후보 문서들(예: 6개)을 먼저 선정
# 2. 각 후보 문서에 대해 Cross-Encoder로 질문-문서 쌍의 점수 계산
# 3. 점수 기준으로 문서 재정렬
# 4. 상위 N개(예: 1개)만 최종 선택

# BGE-Reranker 모델의 특징:
# - 중국어와 영어에 특화되어 있지만, 다국어 능력도 일부 보유
# - 한국어 텍스트에도 비교적 잘 작동할 것으로 기대
# - 모델 크기가 작고 효율적이면서도 좋은 성능을 제공

# top_n 매개변수의 의미:
# - top_n=1: 가장 관련성 높은 문서 1개만 반환 (RAG에서 컨텍스트로 사용하기 적합)
# - top_n=3: 상위 3개 문서 반환 (더 많은 정보를 제공할 때 사용)
# - 사용 사례에 따라 조정 가능: 정확한 답변 필요 시 작은 값, 다양한 정보 필요 시 큰 값

# 리랭킹의 장점:
# 1. 정밀한 관련성 평가: 단순 유사도보다 질문에 답변할 수 있는지 더 잘 평가
# 2. 모호성 해소: "사과"가 과일인지 Apple 기업인지 문맥에 맞게 판단
# 3. 문맥 이해: 질문의 의도와 문서 내용의 연관성을 깊이 있게 분석

# 리랭킹의 비용:
# 1. 계산 비용: 각 문서마다 별도로 모델 실행 필요 (문서 수 × 모델 연산)
# 2. 시간 지연: 추가적인 모델 추론 시간 필요
# 3. 트레이드오프: 정확도 향상 vs 속도 저하

# 실제 적용 시 고려사항:
# 1. 후보 문서 수 조절: 너무 많으면 리랭킹 비용 ↑, 너무 적으면 좋은 문서 놓칠 수 있음
# 2. 점수 임계값: 일정 점수 미만 문서는 필터링 가능
# 3. 다중 모델 활용: 다른 리랭킹 모델과 앙상블 가능

# 다음 단계에서는 이 리랭킹 함수를 실제로 호출하여:
# 1. 하이브리드 검색으로 후보 문서들을 얻고
# 2. Cross-Encoder로 재정렬하여 최종 문서를 선택할 것입니다

# 예상되는 결과:
# - 질문: "서울 강남역 파스타 맛집 추천"
# - 하이브리드 검색 결과: 여러 문서 반환 (일부는 정확히 일치하지 않을 수 있음)
# - 리랭킹 후: 가장 관련성 높은 문서2가 1위로 선정될 것임
# - 문서2: "서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다..."

In [ ]:
# ------------------------------
# 6. 리랭킹 테스트
# ------------------------------
# 테스트용 쿼리(질문)를 정의합니다
# 이 쿼리는 사용자가 서울 강남역 근처의 파스타 맛집을 추천받고 싶어하는 요청입니다
# 쿼리 분석: "서울"(지역) + "강남역"(구체적 위치) + "파스타"(음식 종류) + "맛집 추천"(요청 목적)
query = "서울 강남역 파스타 맛집 추천"

# 1단계: 하이브리드 검색으로 후보 문서 선정
# hybrid_retriever.invoke(query)를 호출하여 BM25와 벡터 검색을 결합한 하이브리드 검색을 실행합니다
# 이 단계에서는 빠른 검색을 통해 질문과 관련 있을 것으로 예상되는 문서들을 후보로 선정합니다
# 후보 문서 수는 하이브리드 검색 함수 내에서 BM25(k=3)와 벡터 검색(k=3)을 결합하므로 최대 6개가 될 수 있습니다
candidate_docs = hybrid_retriever.invoke(query)                         # 1단계: 하이브리드 검색

# 2단계: Cross-Encoder를 사용한 리랭킹
# rerank_with_cross_encoder 함수에 쿼리, 후보 문서들, top_n=1을 전달합니다
# 이 함수는 Cross-Encoder 모델을 사용하여 각 후보 문서와 질문의 관련성을 정밀하게 재평가합니다
# top_n=1로 설정하여 가장 관련성 높은 1개의 문서만 최종 선택합니다
final_docs = rerank_with_cross_encoder(query, candidate_docs, top_n=1)  # 2단계: re-rank

# 최종 선택된 문서(들) 출력
# for 루프를 사용하여 최종 선정된 문서들을 순회하며 내용을 출력합니다
# top_n=1이므로 일반적으로 하나의 문서만 출력됩니다
for d in final_docs:
    print("답변: ", d.page_content)

# 전체 프로세스 상세 설명:
# 1단계 - 하이브리드 검색 동작:
#   - BM25 검색: "서울", "강남역", "파스타", "맛집", "추천" 키워드를 기반으로 문서 검색
#     - 문서2: "서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다..." (모든 키워드 포함)
#     - 문서3: "강남역 맛집 전체 가이드입니다..." (일부 키워드 포함)
#     - 문서4: "서울 홍대 파스타 맛집 추천입니다..." (일부 키워드 포함)
#   - 벡터 검색: 쿼리의 의미적 유사도를 기반으로 문서 검색
#     - 문서2: 강남역 + 파스타 + 맛집의 의미적 유사도 높음
#     - 문서1: 강남역 + 맛집 의미는 있지만 파스타와 직접적 관련 없음
#     - 문서3: 강남역 + 맛집 의미는 있지만 파스타와의 연관성은 다소 낮음
#   - 하이브리드 통합: 두 검색 결과를 가중치 합산하여 최종 후보 문서 리스트 생성
#     - 문서2: 두 검색 모두에서 높은 순위 → 높은 점수(예: 1.0)
#     - 문서3: 두 검색 모두에서 중간 순위 → 중간 점수(예: 0.8)
#     - 문서1,4,5,6: 한쪽 검색에서만 선택 → 낮은 점수(예: 0.5)

# 2단계 - Cross-Encoder 리랭킹 동작:
#   - 후보 문서들과 질문을 쌍으로 묶음:
#     - (질문, 문서2), (질문, 문서3), (질문, 문서1), ...
#   - Cross-Encoder 모델이 각 쌍을 개별적으로 분석:
#     - 모델은 [CLS] 질문 [SEP] 문서 [SEP] 형식으로 입력을 처리
#     - 트랜스포머의 어텐션 메커니즘이 질문과 문서 간의 교차 관계를 분석
#     - 최종 출력층에서 관련성 점수(0-1) 생성
#   - 점수 예상:
#     - 문서2: 0.95 (강남역 + 파스타 + 이탈리안 레스토랑 정확히 일치)
#     - 문서3: 0.70 (강남역 맛집이지만 파스타만 전문적이지 않음)
#     - 문서4: 0.30 (파스타 맛집이지만 홍대 지역)
#     - 문서5: 0.10 (파스타 맛집이지만 부산 지역)
#     - 문서1: 0.05 (강남역이지만 스테이크 전문)
#     - 문서6: 0.02 (강남역이지만 카페)

# 최종 출력 예상:
#   "답변: 서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다. 크림, 토마토 파스타가 유명합니다."

# 리랭킹의 효과 분석:
# 1. 정확도 향상: 단순 키워드/의미 검색보다 더 정확한 문서 선택
# 2. 모호성 해소: "파스타 맛집"을 요청할 때 "스테이크 맛집"이나 "카페"를 배제
# 3. 지역 필터링: "강남역"을 요청할 때 "홍대"나 "부산" 문서 배제

# 실제 애플리케이션에서의 활용:
# - 검색 엔진: 사용자 쿼리에 가장 관련성 높은 문서 우선 표시
# - RAG 시스템: LLM에 제공할 컨텍스트 문서의 품질 향상
# - 추천 시스템: 사용자 질문에 가장 적합한 아이템 추천

# 성능 고려사항:
# - 하이브리드 검색: 빠른 후보 선정 (수 ms ~ 수십 ms)
# - Cross-Encoder 리랭킹: 상대적으로 느림 (문서당 수십 ~ 수백 ms)
# - 최적화: 후보 문서 수 조절, 모델 경량화, 배치 처리 활용

# 확장 가능성:
# 1. 앙상블 리랭킹: 여러 Cross-Encoder 모델의 결과를 조합
# 2. 점수 보정: 검색 점수와 리랭킹 점수를 가중 평균
# 3. 동적 top_n: 쿼리 복잡도에 따라 top_n 값 조정

# 이 테스트는 하이브리드 검색과 리랭킹의 시너지를 보여줍니다:
# - 하이브리드 검색: 폭넓은 후보 선정 (Recall 향상)
# - Cross-Encoder 리랭킹: 정밀한 순위 조정 (Precision 향상)
# - 결과: 더 정확하고 사용자 요구에 맞는 검색 결과 제공

답변:  서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다. 크림, 토마토 파스타가 유명합니다.


In [ ]:
# ------------------------------
# 7. 테스트: RAG vs 하이브리드 vs 리랭크 비교
# ------------------------------
# 동일한 쿼리를 사용하여 세 가지 검색 방법의 결과를 비교합니다
query = "서울 강남역 파스타 맛집 추천"

# (1) RAG 기본: 벡터 검색만 사용
# 전통적인 RAG 시스템에서 가장 일반적으로 사용되는 벡터 검색만 수행합니다
# FAISS 벡터 저장소를 사용하여 쿼리와 의미적 유사도가 높은 문서들을 검색합니다
# k=3으로 설정되어 있으므로 상위 3개의 문서가 반환됩니다
vec_docs = vec.invoke(query)

# (2) 하이브리드 검색: BM25 + 벡터
# 키워드 검색(BM25)과 의미 검색(벡터)을 결합한 하이브리드 접근법입니다
# 두 검색 방법의 결과를 가중치 합산하여 통합 점수를 계산하고 정렬합니다
hybrid_docs = hybrid_retriever.invoke(query)

# (3) 하이브리드 + CrossEncoder 리랭크 (최종 답변에 쓸 문서 1개 선택)
# 가장 정교한 방법: 하이브리드 검색으로 후보 문서를 선정한 후,
# Cross-Encoder 모델을 사용하여 질문과 각 문서의 관련성을 정밀하게 재평가합니다
# top_n=1로 설정하여 가장 관련성 높은 단일 문서만 최종 선택합니다
reranked_docs = rerank_with_cross_encoder(query, hybrid_docs, top_n=1)

# 결과 비교 출력
print("=== 1. 벡터 검색만 사용 (RAG 기본) ===")
# enumerate를 사용하여 순위(1, 2, 3...)와 함께 문서 내용을 출력합니다
# 벡터 검색은 의미적 유사도에 기반하므로 "파스타"라는 단어가 없어도 관련된 문서를 찾을 수 있습니다
for i, d in enumerate(vec_docs, 1):
    print(f"{i}. {d.page_content}")

print("\n=== 2. 하이브리드 검색 (BM25 + 벡터) ===")
# 하이브리드 검색 결과 출력
# 이 방법은 키워드 정확도와 의미적 유사도를 모두 고려하므로 더 균형 잡힌 결과를 제공합니다
for i, d in enumerate(hybrid_docs, 1):
    print(f"{i}. {d.page_content}")

print("\n=== 3. 하이브리드 + CrossEncoder 리랭크 (최종 선택 문서) ===")
# 리랭킹을 거친 최종 문서 출력
# 이 문서는 Cross-Encoder가 질문과 가장 관련성이 높다고 판단한 문서입니다
# RAG 시스템에서는 이 문서를 LLM에 컨텍스트로 제공하여 답변을 생성하게 됩니다
for d in reranked_docs:
    print("답변에 사용할 문서:", d.page_content)

# 세 가지 방법 비교 분석:

# 1. 벡터 검색만 사용 (RAG 기본):
#    - 장점: 구현이 간단하고 빠름, 의미적 유사도 기반으로 관련 문서 찾음
#    - 단점: 정확한 키워드 매칭이 부족할 수 있음, "파스타" 같은 구체적 단어를 놓칠 수 있음
#    - 예상 결과:
#        * 문서2 (강남역 파스타) - 높은 점수
#        * 문서3 (강남역 맛집 가이드) - 중간 점수
#        * 문서1 (강남역 스테이크) - 낮은 점수 (의미적 유사도는 있지만 파스타와 직접적 관련 없음)

# 2. 하이브리드 검색 (BM25 + 벡터):
#    - 장점: 키워드 정확도와 의미적 유사도를 모두 고려, 더 포괄적인 검색
#    - 단점: 두 방법의 가중치 조정이 필요, 계산 비용 증가 (그러나 여전히 빠름)
#    - 예상 결과:
#        * 문서2 (강남역 파스타) - 최고 점수 (두 방법 모두에서 높은 순위)
#        * 문서3 (강남역 맛집 가이드) - 높은 점수
#        * 문서4 (홍대 파스타) - 중간 점수 (파스타 키워드 일치 but 지역 불일치)
#        * 문서1 (강남역 스테이크) - 낮은 점수 (지역은 일치 but 음식 불일치)

# 3. 하이브리드 + CrossEncoder 리랭크:
#    - 장점: 가장 정밀한 관련성 평가, 질문 의도와 문서 내용의 깊은 분석
#    - 단점: 추가적인 계산 비용과 시간 소요, Cross-Encoder 모델 필요
#    - 예상 결과:
#        * 문서2 (강남역 파스타) - 최종 선택 (질문의 모든 요소 완벽히 충족)

# 예상 출력 결과:
# === 1. 벡터 검색만 사용 (RAG 기본) ===
# 1. 서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다...
# 2. 강남역 맛집 전체 가이드입니다. 파스타부터 고기, 한식까지 다양한 식당을 포함합니다.
# 3. 강남역에서 가까운 스테이크 맛집 추천 리스트입니다...
#
# === 2. 하이브리드 검색 (BM25 + 벡터) ===
# 1. 서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다...
# 2. 강남역 맛집 전체 가이드입니다...
# 3. 서울 홍대 파스타 맛집 추천입니다...
#
# === 3. 하이브리드 + CrossEncoder 리랭크 (최종 선택 문서) ===
# 답변에 사용할 문서: 서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다...

# 방법별 선택 기준:
# - 벡터 검색: 의미적 유사도(임베딩 거리)만 고려
# - 하이브리드 검색: 키워드 일치(BM25 점수) + 의미적 유사도(벡터 점수) 통합
# - 리랭킹: 질문-문서 쌍의 심층적 의미 분석(Cross-Encoder 점수)

# 실제 RAG 시스템에서의 적용:
# 1. 단순 RAG: 빠른 응답이 중요하고 정확도 요구가 적을 때
# 2. 하이브리드 RAG: 더 정확한 검색이 필요하고 약간의 추가 비용 감수 가능할 때
# 3. 리랭킹 RAG: 최고의 정확도가 필요하고 비용/지연 시간 감수 가능할 때

# 성능 트레이드오프:
# - 속도: 벡터 > 하이브리드 > 리랭킹
# - 정확도: 리랭킹 > 하이브리드 > 벡터
# - 구현 복잡도: 리랭킹 > 하이브리드 > 벡터

# 최적화 전략:
# - 스테이지드 검색: 1) 빠른 벡터 검색으로 100개 후보 → 2) 하이브리드로 20개 선정 → 3) 리랭킹으로 3개 최종
# - 캐싱: 자주 묻는 질문의 검색 결과 캐싱
# - 점수 임계값: 일정 점수 미만 문서는 RAG 컨텍스트에서 제외

# 이 비교를 통해 다양한 검색 기법의 장단점을 이해하고,
# 사용 사례에 맞는 최적의 검색 전략을 선택할 수 있습니다.

=== 1. 벡터 검색만 사용 (RAG 기본) ===
1. 서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다. 크림, 토마토 파스타가 유명합니다.
2. 서울 홍대 파스타 맛집 추천입니다. 강남역과는 거리가 꽤 있습니다.
3. 강남역 맛집 전체 가이드입니다. 파스타부터 고기, 한식까지 다양한 식당을 포함합니다.

=== 2. 하이브리드 검색 (BM25 + 벡터) ===
1. 서울 홍대 파스타 맛집 추천입니다. 강남역과는 거리가 꽤 있습니다.
2. 서울 강남역 카페 추천 리스트입니다. 디저트와 커피로 유명한 곳이 많습니다.
3. 부산 남천동 파스타 맛집 리스트입니다. 부산 지역 이탈리안 맛집 모음입니다.
4. 서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다. 크림, 토마토 파스타가 유명합니다.
5. 강남역 맛집 전체 가이드입니다. 파스타부터 고기, 한식까지 다양한 식당을 포함합니다.

=== 3. 하이브리드 + CrossEncoder 리랭크 (최종 선택 문서) ===
답변에 사용할 문서: 서울 강남역에서 파스타가 맛있는 이탈리안 레스토랑 리스트입니다. 크림, 토마토 파스타가 유명합니다.
